# 12 · Mock coding test B — binary classification, class imbalance, and a two-task network

**Rules.** Same as mock A: **150 minutes**, one sitting, no autocomplete / internet / AI / other notebooks; allowed are your homework, the official course materials and `help(...)`.

This mock deliberately changes the details you have been practising, to test whether you understood the ideas or memorised one version of them:

- the file is parsed with the `csv` module,
- missing values are filled with the **mean** and flagged with **1 = missing**,
- features are **min-max** scaled instead of standardised,
- mini-batches come from **your own generator** — `Dataset`/`DataLoader` are *not allowed* here,
- the classifier is built the **lecturer's way** (sigmoid in the model, `BCELoss`),
- the classes are imbalanced, and the last task trains regression and classification **together**.

`check(...)` lines print ✅ / ❌. For the training tasks they only test loose sanity ranges.

---
**The data.** `machines.csv` — sensor snapshots of industrial machines, comma separated, with a header:

`temp_c, vibration, pressure, rpm, hours_since_service, type, shift, wear, failure`

- `type` is `A`, `B` or `C`; `shift` is `day` or `night`
- `vibration` and `pressure` are sometimes missing (empty field)
- `wear` (0–100) is measured later in the workshop — a **label**
- `failure` (0/1): did the machine fail in the following week — a **label**

In [ ]:
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import sys; sys.path.insert(0, "..")
from dnd_utils import check, DATA, reference_mock_b

REF = reference_mock_b()
PATH = DATA / "machines.csv"
with open(PATH) as f:
    for _ in range(4): print(f.readline().rstrip())

## **1** · Load (15 min)

With `csv.reader`, build `raw`: shape `(2400, 9)`, `float32`, columns in file order; `type` → 0/1/2 for A/B/C; `shift` → 0 for day, 1 for night; empty → `np.nan`.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE
check("1 · raw", lambda: raw.shape == (2400, 9) and raw.dtype == np.float32 and np.allclose(raw, REF["raw"], equal_nan=True))

## **2** · Shuffle and split (5 min)

Permutation `np.random.default_rng(7).permutation(n)`; split **70 % / 15 % / 15 %** (`int(n * 0.7)`, `int(n * 0.15)`, rest) into `split_train`, `split_val`, `split_test`.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE
check("2 · splits", lambda: all(a.shape == b.shape and np.allclose(a, b, equal_nan=True) for a, b in zip((split_train, split_val, split_test), REF["splits"])))

## **3** · Pre-process (20 min)

Build `X_train`, `X_val`, `X_test` (`float32`, **11 columns**):

| columns | content |
|---|---|
| 0–4 | `temp_c`, `vibration`, `pressure`, `rpm`, `hours_since_service` — missing values replaced by the **training mean** of the column, then **min-max scaled** to `[0, 1]` using the training minimum and maximum (computed after the replacement) |
| 5 | `shift` (0/1) |
| 6–8 | one-hot of `type` (A, B, C) |
| 9–10 | indicators for `vibration` and `pressure`: **1 if the value was missing**, else 0 |

Also create the label vectors `wear_train/val/test` and `fail_train/val/test` (1-D, `float32`). Neither label may appear among the inputs.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE
check("3 · X matrices", lambda: all(a.shape == b.shape and a.dtype == np.float32 and np.allclose(a, b, atol=1e-5) for a, b in zip((X_train, X_val, X_test), REF["X"])))
check("3 · training features lie exactly in [0, 1]", lambda: X_train[:, :5].min() == 0 and X_train[:, :5].max() == 1)
check("3 · labels", lambda: all(np.array_equal(a, b) for a, b in zip((wear_train, wear_val, wear_test, fail_train, fail_val, fail_test), REF["wear"] + REF["fail"])))

## **4** · A mini-batch generator (15 min)

Write the generator function `batch_generator(X, labels, batch_size, shuffle, rng=None)`:

- `X`: NumPy `(n, d)`; `labels`: a **list** of one or more 1-D NumPy arrays of length `n` (one per task)
- yields tuples `(xb, l1b, l2b, …)` of `torch.float32` tensors: `xb` `(batch, d)`, every label batch `(batch, 1)`
- `shuffle=True` → a new random order from `rng` on every call; `shuffle=False` → original order

`Dataset` and `DataLoader` are not allowed in this notebook.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE
_g = list(batch_generator(X_train, [fail_train, wear_train], 64, False))
check("4 · number of batches and shapes", lambda: len(_g) == 27 and [tuple(t.shape) for t in _g[0]] == [(64, 11), (64, 1), (64, 1)] and _g[-1][0].shape[0] == 1680 - 26 * 64)
check("4 · dtypes", lambda: all(t.dtype == torch.float32 for t in _g[0]))
check("4 · unshuffled keeps the order", lambda: np.array_equal(torch.cat([b[2] for b in _g])[:, 0].numpy(), wear_train))
_s = list(batch_generator(X_train, [wear_train], 64, True, np.random.default_rng(0)))
check("4 · shuffled: other order, same samples, pairs intact", lambda: not np.array_equal(torch.cat([b[1] for b in _s])[:, 0].numpy(), wear_train) and np.allclose(np.sort(torch.cat([b[1] for b in _s])[:, 0].numpy()), np.sort(wear_train)) and any(np.array_equal(_s[0][0][0].numpy(), X_train[i]) for i in np.where(wear_train == _s[0][1][0, 0].item())[0]))

## **5** · Failure classifier with early stopping (35 min)

- Model class `FailureMLP`: `11 → 32 → 16 → 1`, ReLU, **sigmoid at the output**; loss `nn.BCELoss`.
- Train with your generator (batch size 64, new shuffle each epoch), `Adam`. Record training and validation loss per epoch.
- Early stopping on the validation loss, `patience = 15`, at most 300 epochs, best weights restored automatically.
- Plot both curves with a legend and a marker at the best epoch.
- On the test set compute `test_acc`, `test_precision`, `test_recall`, `test_f1` (positive class = failure; guard against division by zero) and `baseline_acc` (always predict the training majority class).
- One comment line: is accuracy a good metric here? Why / why not?

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE
check("5 · model outputs probabilities of shape (batch, 1)", lambda: (lambda o: o.shape == (7, 1) and 0 <= o.min() and o.max() <= 1)(model(torch.rand(7, 11))))
check("5 · baseline_acc", lambda: abs(baseline_acc - (fail_test == 0).mean()) < 1e-6)
check("5 · metrics are consistent", lambda: abs(test_f1 - (2 * test_precision * test_recall / (test_precision + test_recall) if test_precision + test_recall > 0 else 0.0)) < 1e-6)
check("5 · the classifier learned something (accuracy ≥ baseline, recall > 0.2)", lambda: test_acc >= baseline_acc and test_recall > 0.2)

## **6** · A weighted loss against the imbalance (20 min)

Missing a failure is expensive. Train a second `FailureMLP` (`model_w`) in which the loss of every **failure** sample is multiplied by `w_pos = n_negative / n_positive` (training set). Use `nn.BCELoss(reduction="none")` and do the weighting and averaging yourself. Early stopping as before, **monitoring the plain (unweighted) validation BCE**.

Report `recall_w` and `precision_w` on the test set next to the values from task 5, and comment in one line on what changed and why.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE
check("6 · w_pos", lambda: abs(w_pos - (fail_train == 0).sum() / (fail_train == 1).sum()) < 1e-6)
check("6 · weighting did not lower the recall (normally it rises)", lambda: recall_w >= test_recall)
check("6 · ... and did not raise the precision (normally it falls)", lambda: precision_w <= test_precision + 0.02)

## **7** · One network, two tasks (40 min)

Predict `wear` (regression) **and** `failure` (classification) with a single network.

- Standardise the `wear` target with training statistics.
- `TwoTaskMLP`: shared layers `11 → 32 → 16` (ReLU), then two heads: `wear` (1 output) and `failure` (1 **logit** — use `nn.BCEWithLogitsLoss` here). `forward` returns both.
- Loss: `MSE(wear) + BCE(failure)`. Train 100 epochs with your generator (it already supports two label arrays), `Adam`; record the total training and validation loss per epoch and plot them.
- On the test set report `wear_rmse` **in the original wear units**, `wear_baseline_rmse` (predicting the training mean), and `fail_acc`.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE
check("7 · two outputs of shape (batch, 1)", lambda: (lambda o: len(o) == 2 and o[0].shape == (5, 1) and o[1].shape == (5, 1))(model_mt(torch.rand(5, 11))))
check("7 · baseline RMSE", lambda: abs(wear_baseline_rmse - float(np.sqrt(((wear_test - wear_train.mean()) ** 2).mean()))) < 1e-3)
check("7 · wear RMSE is in wear units and far below the baseline", lambda: 2 < wear_rmse < 0.5 * wear_baseline_rmse)
check("7 · failure accuracy at least at baseline level", lambda: fail_acc >= baseline_acc - 0.02)

---
## After the bell

For each task write one line: *finished / finished with docs / not finished*, and the **single idea** that slowed you down most. Typical ones, and where to revise them:

| if this cost you time … | … revisit |
|---|---|
| `csv.reader`, header handling | 07, exercise 7.1 |
| fitting fill values / scaling on the training split only | 04, exercises 4.6–4.9 |
| generator mechanics, re-creating it every epoch | 05, exercises 5.3–5.4 |
| sigmoid + `BCELoss` vs logits + `BCEWithLogitsLoss` | 07, exercises 7.3–7.4, 7.10 |
| early stopping, `deepcopy` of the state | 09, exercises 9.2–9.3 |
| `reduction="none"` and weight shapes | 07, exercise 7.7 |
| two heads, loss scales, un-standardising predictions | 10 |

Then, a few days later, repeat the mock from an empty copy and try to beat your time.